In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, classification_report
from collections import Counter, defaultdict
from datasets import load_dataset
import random
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

c:\Users\Dell\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load SMS Spam dataset
print('Loading SMS Spam dataset...')
sms_data = load_dataset('ucirvine/sms_spam', split='train')

# Shuffle and split
sms_data = sms_data.shuffle(seed=42)
split_idx = int(0.8 * len(sms_data))

train_sms = sms_data.select(range(split_idx))
test_sms = sms_data.select(range(split_idx, len(sms_data)))

print(f'\nDataset: SMS Spam Collection')
print(f'Train: {len(train_sms)} messages')
print(f'Test: {len(test_sms)} messages')
print(f'\nClasses: 0=Not Spam, 1=Spam')

print(f'\nSample messages:')
print(f'Message: {train_sms[0]["sms"]}')
print(f'Label: {train_sms[0]["label"]}')
print(f'\nMessage: {train_sms[10]["sms"]}')
print(f'Label: {train_sms[10]["label"]}')

Loading SMS Spam dataset...



Dataset: SMS Spam Collection
Train: 4459 messages
Test: 1115 messages

Classes: 0=Not Spam, 1=Spam

Sample messages:
Message: sports fans - get the latest sports news str* 2 ur mobile 1 wk FREE PLUS a FREE TONE Txt SPORT ON to 8007 www.getzed.co.uk 0870141701216+ norm 4txt/120p 

Label: 1

Message: Cramps stopped. Going back to sleep

Label: 0


In [10]:
all_sms = train_sms['sms']     
all_labels = train_sms['label'] 
combined = list(zip(all_sms, all_labels))
random.shuffle(combined)
all_sms, all_labels = zip(*combined)
all_sms = list(all_sms)
all_labels = list(all_labels)

all_sms_test = test_sms['sms']
all_labels_test = test_sms['label']
combined_test = list(zip(all_sms_test, all_labels_test))
random.shuffle(combined_test)
all_sms_test, all_labels_test = zip(*combined_test)
all_sms_test = list(all_sms_test)
all_labels_test = list(all_labels_test)

In [11]:
vocab = {'<PAD>': 0, '<UNK>': 1}
word_counts = Counter()

for sms in all_sms:
    words = sms.split()
    word_counts.update(words)

for word, count in word_counts.items():
    if count >= 2 and word not in vocab:
        vocab[word] = len(vocab)

print(f"Vocabulary size: {len(vocab)}")
print(f"Sample vocab: {list(vocab.items())[:10]}")

id2word = {idx: word for word, idx in vocab.items()}

Vocabulary size: 5445
Sample vocab: [('<PAD>', 0), ('<UNK>', 1), ('To', 2), ('review', 3), ('and', 4), ('KEEP', 5), ('the', 6), ('fantastic', 7), ('Nokia', 8), ('game', 9)]


In [12]:
EMBEDDING_DIM = 100

glove_path = 'glove_vectors.txt'

glove_vectors = {}
skipped = 0

with open(glove_path, 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(f):
        try:
            values = line.rstrip().split()
            word = values[0]
            vector = np.array(values[1:], dtype='float32')

            if len(vector) == EMBEDDING_DIM:
                glove_vectors[word] = vector
            else:
                skipped += 1
        except (ValueError, IndexError):
                skipped += 1
                continue

        if (line_num + 1) % 100000 == 0:
                    print(f"  Processed {line_num + 1:,} lines...")

print(f"\nLoaded {len(glove_vectors):,} GloVe vectors")
if skipped > 0:
    print(f"Skipped {skipped:,} malformed lines")

  Processed 100,000 lines...
  Processed 200,000 lines...
  Processed 300,000 lines...
  Processed 400,000 lines...
  Processed 500,000 lines...
  Processed 600,000 lines...
  Processed 700,000 lines...
  Processed 800,000 lines...
  Processed 900,000 lines...
  Processed 1,000,000 lines...
  Processed 1,100,000 lines...
  Processed 1,200,000 lines...

Loaded 1,287,614 GloVe vectors
Skipped 3,533 malformed lines


In [13]:
embedding_matrix = np.random.rand(len(vocab), EMBEDDING_DIM).astype(np.float32)
embedding_matrix[vocab['<PAD>']] = np.zeros(EMBEDDING_DIM)

found_count = 0
for word, idx in vocab.items():
    if word in glove_vectors:
        embedding_matrix[idx] = glove_vectors[word]
        found_count += 1

print(f"\nEmbedding matrix shape: {embedding_matrix.shape}")
print(f"Found pre-trained vectors for {found_count}/{len(vocab)} words ({100*found_count/len(vocab):.1f}%)")


Embedding matrix shape: (5445, 100)
Found pre-trained vectors for 2381/5445 words (43.7%)


In [ ]:
class SMSDataset(Dataset):
    def __init__(self, sms_list, label_list, vocab):
        self.sms_list = sms_list
        self.label_list = label_list
        self.vocab = vocab

    def __len__(self):
        return len(self.sms_list)

    def __getitem__(self, idx):
        sms = self.sms_list[idx]
        label = self.label_list[idx]

        words = sms.split()

        sms_indices = [
            self.vocab.get(word, self.vocab['<UNK>'])
            for word in words]

        return (
            torch.tensor(sms_indices, dtype=torch.long),
            torch.tensor(label, dtype=torch.long))

def collate_fn(batch):
    sentences, labels = zip(*batch)

    lengths = torch.tensor(
        [len(sentence) for sentence in sentences],
        dtype=torch.long)

    padded_sentences = pad_sequence(
        sentences,
        batch_first=True,
        padding_value=vocab['<PAD>'])

    labels = torch.stack(labels)

    return padded_sentences, labels, lengths


train_dataset = SMSDataset(
    all_sms,
    all_labels,
    vocab)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn)


test_dataset = SMSDataset(
    all_sms_test,
    all_labels_test,
    vocab)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn)

In [5]:
from torch.nn.utils.rnn import pack_padded_sequence

class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, embedding_matrix):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))

        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            batch_first=True)

        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, lengths):
        embedded = self.embedding(x)

        packed_embedded = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False)

        _, hidden = self.rnn(packed_embedded)

        hidden = hidden.squeeze(0)
        predictions = self.fc(hidden)

        return predictions


# GRU Model
class GRUModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, embedding_matrix):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))

        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            batch_first=True)

        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, lengths):
        embedded = self.embedding(x)

        packed_embedded = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False)

        _, hidden = self.gru(packed_embedded)

        hidden = hidden.squeeze(0)
        predictions = self.fc(hidden)
        return predictions


# LSTM Model
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, embedding_matrix):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True)

        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, lengths):
        embedded = self.embedding(x)

        packed_embedded = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False)

        _, (hidden, _) = self.lstm(packed_embedded)

        hidden = hidden.squeeze(0)

        predictions = self.fc(hidden)

        return predictions


# BiLSTM Model
class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, embedding_matrix):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True)

        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x, lengths):
        embedded = self.embedding(x)

        packed_embedded = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False )

        _, (hidden, _) = self.lstm(packed_embedded)

        # hidden shape:
        # [2, batch_size, hidden_dim]
        # hidden[0] = forward final hidden state
        # hidden[1] = backward final hidden state

        forward_hidden = hidden[0]
        backward_hidden = hidden[1]

        hidden = torch.cat(
            (forward_hidden, backward_hidden),
            dim=1
        )

        predictions = self.fc(hidden)
        return predictions

In [21]:
def train_spam_model(model, train_loader, epochs=10, lr=0.001):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses = []
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for texts, labels, lengths in train_loader:
            texts, labels = texts.to(device), labels.to(device)
            optimizer.zero_grad()
            predictions = model(texts, lengths)
            loss = criterion(predictions, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        losses.append(avg_loss)
        print(f'Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}')
    return losses


def evaluate_spam_model(model, test_loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for texts, labels, lengths in test_loader:
            texts = texts.to(device)
            predictions = model(texts, lengths)
            predicted_labels = torch.argmax(predictions, dim=1).cpu().numpy()
            all_preds.extend(predicted_labels)
            all_labels.extend(labels.numpy())
    accuracy = accuracy_score(all_labels, all_preds)
    return accuracy


print('Training functions ready!')

Training functions ready!


In [ ]:
# Train all 4 spam models
SPAM_HIDDEN_DIM = 128
SPAM_EPOCHS = 10
SPAM_LR = 0.001
SPAM_OUTPUT_DIM = 2

spam_results = {}

# Vanilla RNN
print('=' * 60)
print('Training Vanilla RNN for Spam...')
print('=' * 60)

rnn_spam = VanillaRNN(
    len(vocab),
    EMBEDDING_DIM,
    SPAM_HIDDEN_DIM,
    SPAM_OUTPUT_DIM,
    embedding_matrix)

rnn_spam_losses = train_spam_model(
    rnn_spam,
    train_loader,
    epochs=SPAM_EPOCHS,
    lr=SPAM_LR)

rnn_spam_acc = evaluate_spam_model(
    rnn_spam,
    test_loader)

spam_results['Vanilla RNN'] = {
    'losses': rnn_spam_losses,
    'accuracy': rnn_spam_acc}

print(f'\nVanilla RNN Spam Accuracy: {rnn_spam_acc:.4f}\n')


# GRU
print('=' * 60)
print('Training GRU for Spam...')
print('=' * 60)

gru_spam = GRUModel(
    len(vocab),
    EMBEDDING_DIM,
    SPAM_HIDDEN_DIM,
    SPAM_OUTPUT_DIM,
    embedding_matrix)

gru_spam_losses = train_spam_model(
    gru_spam,
    train_loader,
    epochs=SPAM_EPOCHS,
    lr=SPAM_LR)

gru_spam_acc = evaluate_spam_model(
    gru_spam,
    test_loader)

spam_results['GRU'] = {
    'losses': gru_spam_losses,
    'accuracy': gru_spam_acc}

print(f'\nGRU Spam Accuracy: {gru_spam_acc:.4f}\n')


# LSTM
print('=' * 60)
print('Training LSTM for Spam...')
print('=' * 60)

lstm_spam = LSTMModel(
    len(vocab),
    EMBEDDING_DIM,
    SPAM_HIDDEN_DIM,
    SPAM_OUTPUT_DIM,
    embedding_matrix)

lstm_spam_losses = train_spam_model(
    lstm_spam,
    train_loader,
    epochs=SPAM_EPOCHS,
    lr=SPAM_LR)

lstm_spam_acc = evaluate_spam_model(
    lstm_spam,
    test_loader)

spam_results['LSTM'] = {
    'losses': lstm_spam_losses,
    'accuracy': lstm_spam_acc}

print(f'\nLSTM Spam Accuracy: {lstm_spam_acc:.4f}\n')


# BiLSTM
print('=' * 60)
print('Training BiLSTM for Spam...')
print('=' * 60)

bilstm_spam = BiLSTMModel(
    len(vocab),
    EMBEDDING_DIM,
    SPAM_HIDDEN_DIM,
    SPAM_OUTPUT_DIM,
    embedding_matrix)

bilstm_spam_losses = train_spam_model(
    bilstm_spam,
    train_loader,
    epochs=SPAM_EPOCHS,
    lr=SPAM_LR)

bilstm_spam_acc = evaluate_spam_model(
    bilstm_spam,
    test_loader)

spam_results['BiLSTM'] = {
    'losses': bilstm_spam_losses,
    'accuracy': bilstm_spam_acc}

print(f'\nBiLSTM Spam Accuracy: {bilstm_spam_acc:.4f}\n')

Training Vanilla RNN for Spam...
Epoch 1/10, Loss: 0.2440
Epoch 2/10, Loss: 0.1314
Epoch 3/10, Loss: 0.0897
Epoch 4/10, Loss: 0.0511
Epoch 5/10, Loss: 0.0934
Epoch 6/10, Loss: 0.0365
Epoch 7/10, Loss: 0.0270
Epoch 8/10, Loss: 0.0172
Epoch 9/10, Loss: 0.0131
Epoch 10/10, Loss: 0.0142

Vanilla RNN Spam Accuracy: 0.9821

Training GRU for Spam...
Epoch 1/10, Loss: 0.2248
Epoch 2/10, Loss: 0.0928
Epoch 3/10, Loss: 0.0473
Epoch 4/10, Loss: 0.0300
Epoch 5/10, Loss: 0.0230
Epoch 6/10, Loss: 0.0121
Epoch 7/10, Loss: 0.0115
Epoch 8/10, Loss: 0.0051
Epoch 9/10, Loss: 0.0032
Epoch 10/10, Loss: 0.0020

GRU Spam Accuracy: 0.9874

Training LSTM for Spam...
Epoch 1/10, Loss: 0.2474
Epoch 2/10, Loss: 0.0961
Epoch 3/10, Loss: 0.0522
Epoch 4/10, Loss: 0.0366
Epoch 5/10, Loss: 0.0231
Epoch 6/10, Loss: 0.0126
Epoch 7/10, Loss: 0.0176
Epoch 8/10, Loss: 0.0166
Epoch 9/10, Loss: 0.0113
Epoch 10/10, Loss: 0.0052

LSTM Spam Accuracy: 0.9874

Training BiLSTM for Spam...
Epoch 1/10, Loss: 0.2353
Epoch 2/10, Loss:

In [24]:
import torch
import torch.nn.functional as F

def predict_spam(model, message):
    model.eval()

    # Tokenize the message using the same method as training
    words = message.split()

    # Convert words to IDs using the training vocabulary
    indices = [
        vocab.get(word, vocab['<UNK>'])
        for word in words
    ]

    # Handle an empty message
    if len(indices) == 0:
        print("Please enter a non-empty message.")
        return

    # Create input tensor: [1, sequence_length]
    x = torch.tensor(indices, dtype=torch.long).unsqueeze(0).to(device)

    # Actual message length
    lengths = torch.tensor([len(indices)], dtype=torch.long)

    # Predict
    with torch.no_grad():
        logits = model(x, lengths)
        probabilities = F.softmax(logits, dim=1)

    predicted_class = probabilities.argmax(dim=1).item()

    labels = {
        0: "Not Spam",
        1: "Spam"
    }

    print(f"\nMessage: {message}")
    print(f"Prediction: {labels[predicted_class]}")
    print(f"Not Spam probability: {probabilities[0, 0].item() * 100:.2f}%")
    print(f"Spam probability: {probabilities[0, 1].item() * 100:.2f}%")


# Select the model you want to test
models = {
    "Vanilla RNN": rnn_spam,
    "GRU": gru_spam,
    "LSTM": lstm_spam,
    "BiLSTM": bilstm_spam
}

message = input("Enter an SMS message: ")

for model_name, model in models.items():
    print(f"\n{'=' * 40}")
    print(model_name)
    print('=' * 40)
    predict_spam(model, message)


Vanilla RNN

Message: Hey, are we still meeting tomorrow? Let me know what time works for you.
Prediction: Not Spam
Not Spam probability: 99.86%
Spam probability: 0.14%

GRU

Message: Hey, are we still meeting tomorrow? Let me know what time works for you.
Prediction: Not Spam
Not Spam probability: 100.00%
Spam probability: 0.00%

LSTM

Message: Hey, are we still meeting tomorrow? Let me know what time works for you.
Prediction: Not Spam
Not Spam probability: 99.99%
Spam probability: 0.01%

BiLSTM

Message: Hey, are we still meeting tomorrow? Let me know what time works for you.
Prediction: Not Spam
Not Spam probability: 100.00%
Spam probability: 0.00%


In [25]:
import torch

checkpoint = {
    'model_state_dict': bilstm_spam.state_dict(),
    'vocab': vocab,
    'embedding_matrix': embedding_matrix,
    'vocab_size': len(vocab),
    'embedding_dim': EMBEDDING_DIM,
    'hidden_dim': SPAM_HIDDEN_DIM,
    'output_dim': SPAM_OUTPUT_DIM
}

torch.save(checkpoint, 'bilstm_spam.pth')

print("BiLSTM model saved successfully!")

BiLSTM model saved successfully!


## BiLSTM model

In [9]:
import torch
import torch.nn.functional as F

class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, embedding_matrix):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True)

        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x, lengths):
        embedded = self.embedding(x)

        packed_embedded = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False )

        _, (hidden, _) = self.lstm(packed_embedded)

        # hidden shape:
        # [2, batch_size, hidden_dim]
        # hidden[0] = forward final hidden state
        # hidden[1] = backward final hidden state

        forward_hidden = hidden[0]
        backward_hidden = hidden[1]

        hidden = torch.cat(
            (forward_hidden, backward_hidden),
            dim=1
        )

        predictions = self.fc(hidden)
        return predictions

checkpoint = torch.load(
    'bilstm_spam.pth',
    map_location=device,
    weights_only=False
)

vocab = checkpoint['vocab']
embedding_matrix = checkpoint['embedding_matrix']

bilstm_spam = BiLSTMModel(
    checkpoint['vocab_size'],
    checkpoint['embedding_dim'],
    checkpoint['hidden_dim'],
    checkpoint['output_dim'],
    embedding_matrix
).to(device)

bilstm_spam.load_state_dict(checkpoint['model_state_dict'])
bilstm_spam.eval()

print("BiLSTM model loaded successfully!")

def predict_spam(model, message):
    model.eval()

    words = message.split()

    indices = [
        vocab.get(word, vocab['<UNK>'])
        for word in words
    ]

    if len(indices) == 0:
        print("Please enter a non-empty message.")
        return

    x = torch.tensor(indices, dtype=torch.long).unsqueeze(0).to(device)

    lengths = torch.tensor([len(indices)], dtype=torch.long)

    with torch.no_grad():
        logits = model(x, lengths)
        probabilities = F.softmax(logits, dim=1)

    predicted_class = probabilities.argmax(dim=1).item()

    labels = {
        0: "Not Spam",
        1: "Spam"
    }

    print(f"\nMessage: {message}")
    print(f"Prediction: {labels[predicted_class]}")
    print(f"Not Spam probability: {probabilities[0, 0].item() * 100:.2f}%")
    print(f"Spam probability: {probabilities[0, 1].item() * 100:.2f}%")


models = {
    "BiLSTM": bilstm_spam
}

message = input("Enter an SMS message: ")

for model_name, model in models.items():
    print(f"\n{'=' * 40}")
    print(model_name)
    print('=' * 40)
    predict_spam(model, message)

BiLSTM model loaded successfully!

BiLSTM

Message: Hey, are we still meeting tomorrow? Let me know what time works for you.
Prediction: Not Spam
Not Spam probability: 100.00%
Spam probability: 0.00%
